# Spirit Airlines Fare Impact Analysis

**Objective:** Analyze fare impact of Spirit Airlines shutdown (May 2, 2026) on 20 busiest routes.

**Data:** BTS Q1-Q2 2025 vs. Google Flights May 2026 snapshot

**Questions to Answer:**
1. How much did fares rise on Spirit's routes?
2. Which routes were hit hardest?
3. Which carriers filled the gap? (especially Frontier)
4. Did market concentration increase?

## 1. Load Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Define data path
data_path = Path('../data/processed')

print('✅ Libraries loaded')

✅ Libraries loaded


## 2. Load BTS Data (Q1 & Q2 2025)

In [2]:
# Load Q1 and Q2 BTS data
bts_q1 = pd.read_csv(data_path / 'spirit_20routes_clean.csv')
bts_q2 = pd.read_csv(data_path / 'spirit_20routes_clean_q2.csv')

print(f'Q1 rows: {len(bts_q1)}')
print(f'Q2 rows: {len(bts_q2)}')
print(f'\nQ1 Head:')
print(bts_q1.head())
print(f'\nQ2 Head:')
print(bts_q2.head())

Q1 rows: 15939
Q2 rows: 14010

Q1 Head:
   YEAR  QUARTER ORIGIN DEST OPERATING_CARRIER  PASSENGERS  MARKET_FARE
0  2025        1    FLL  LGA                NK         1.0         5.50
1  2025        1    FLL  LGA                NK         1.0         5.50
2  2025        1    FLL  LGA                NK         1.0         5.53
3  2025        1    FLL  LGA                NK         1.0        14.57
4  2025        1    FLL  LGA                NK         1.0        15.00

Q2 Head:
   YEAR  QUARTER ORIGIN DEST OPERATING_CARRIER  PASSENGERS  MARKET_FARE
0  2025        2    FLL  LGA                NK         1.0         4.68
1  2025        2    FLL  LGA                NK         1.0         5.53
2  2025        2    FLL  LGA                NK         1.0        16.00
3  2025        2    FLL  LGA                NK         1.0        19.60
4  2025        2    FLL  LGA                NK         1.0        21.00


## 3. Combine BTS Q1 + Q2 & Explore

In [3]:
# Combine Q1 and Q2 (all quarters have Spirit carrier code NK)
bts_combined = pd.concat([bts_q1, bts_q2], ignore_index=True)

# Filter to Spirit Airlines only (NK)
bts_spirit = bts_combined[bts_combined['OPERATING_CARRIER'] == 'NK'].copy()

print(f'Combined BTS rows: {len(bts_combined)}')
print(f'Spirit (NK) only rows: {len(bts_spirit)}')
print(f'\nQuarters in data: {sorted(bts_spirit["QUARTER"].unique())}')
print(f'Unique routes: {len(bts_spirit.groupby(["ORIGIN", "DEST"]))}')
print(f'\nBTS Data Info:')
print(bts_spirit.info())

Combined BTS rows: 29949
Spirit (NK) only rows: 29949

Quarters in data: [np.int64(1), np.int64(2)]
Unique routes: 20

BTS Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29949 entries, 0 to 29948
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   YEAR               29949 non-null  int64  
 1   QUARTER            29949 non-null  int64  
 2   ORIGIN             29949 non-null  object 
 3   DEST               29949 non-null  object 
 4   OPERATING_CARRIER  29949 non-null  object 
 5   PASSENGERS         29949 non-null  float64
 6   MARKET_FARE        29949 non-null  float64
dtypes: float64(2), int64(2), object(3)
memory usage: 1.6+ MB
None


## 4. Load Google Flights Data (May 2026)

In [4]:
# Load Google Flights data
gf_data = pd.read_csv(data_path / 'google_flights_fares.csv')

print(f'Google Flights rows: {len(gf_data)}')
print(f'\nGoogle Flights Head:')
print(gf_data.head(10))
print(f'\nData types:')
print(gf_data.dtypes)
print(f'\nUnique carriers in Google Flights data:')
print(gf_data['Carrier'].unique())

Google Flights rows: 20

Google Flights Head:
                           Route           Origin  Destination    Search_Date  \
0     Fort Lauderdale → New York  Fort Lauderdale     New York  May 8th, 2026   
1      Fort Lauderdale → Chicago  Fort Lauderdale      Chicago  May 8th, 2026   
2      Fort Lauderdale → Atlanta  Fort Lauderdale      Atlanta  May 8th, 2026   
3       Fort Lauderdale → Dallas  Fort Lauderdale       Dallas  May 8th, 2026   
4  Fort Lauderdale → Los Angeles  Fort Lauderdale  Los Angeles  May 8th, 2026   
5             Orlando → New York          Orlando     New York  May 8th, 2026   
6              Orlando → Chicago          Orlando      Chicago  May 8th, 2026   
7            Orlando → Baltimore          Orlando    Baltimore  May 8th, 2026   
8             Las Vegas → Dallas        Las Vegas       Dallas  May 8th, 2026   
9        Las Vegas → Los Angeles        Las Vegas  Los Angeles  May 8th, 2026   

       Travel_Date Current_fare   Carrier  
0  June 15th, 2026

## 5. Clean Google Flights Data

In [5]:
# Clean Current_fare: remove $ and convert to numeric
gf_data['Current_fare_clean'] = gf_data['Current_fare'].str.replace('$', '').astype(float)

# Standardize origin/dest columns (use airport codes)
# Extract from Origin/Destination or use existing columns
# Assuming Origin/Destination columns have full city names, let's map to codes

# For now, let's check the structure
print(gf_data[['Origin', 'Destination', 'Current_fare_clean', 'Carrier']].head(10))
print(f'\nFare range: ${gf_data["Current_fare_clean"].min():.2f} - ${gf_data["Current_fare_clean"].max():.2f}')
print(f'Average fare: ${gf_data["Current_fare_clean"].mean():.2f}')

            Origin  Destination  Current_fare_clean   Carrier
0  Fort Lauderdale     New York                99.0     Delta
1  Fort Lauderdale      Chicago               134.0    United
2  Fort Lauderdale      Atlanta                56.0  Frontier
3  Fort Lauderdale       Dallas               147.0  Frontier
4  Fort Lauderdale  Los Angeles               153.0   JetBlue
5          Orlando     New York               108.0  Frontier
6          Orlando      Chicago               142.0  Frontier
7          Orlando    Baltimore               115.0  Frontier
8        Las Vegas       Dallas               146.0  Frontier
9        Las Vegas  Los Angeles                84.0  Frontier

Fare range: $26.00 - $217.00
Average fare: $126.95


## 6. Create Origin/Destination Airport Codes Mapping

In [6]:
# Mapping of city names to airport codes (based on the 20 routes)
city_to_code = {
    'Fort Lauderdale': 'FLL',
    'Orlando': 'MCO',
    'Las Vegas': 'LAS',
    'Houston': 'IAH',
    'Detroit': 'DTW',
    'Baltimore': 'BWI',
    'Philadelphia': 'PHL',
    'Atlanta': 'ATL',
    'New York': 'LGA',
    'Chicago': 'ORD',
    'Dallas': 'DFW',
    'Los Angeles': 'LAX',
}

# Apply mapping
gf_data['Origin_Code'] = gf_data['Origin'].str.strip().map(city_to_code)
gf_data['Dest_Code'] = gf_data['Destination'].str.strip().map(city_to_code)

# Create route identifier
gf_data['Route'] = gf_data['Origin_Code'] + '-' + gf_data['Dest_Code']

print('Sample with codes:')
print(gf_data[['Origin', 'Origin_Code', 'Destination', 'Dest_Code', 'Route', 'Current_fare_clean', 'Carrier']].head(10))
print(f'\nUnmapped Origins: {gf_data[gf_data["Origin_Code"].isna()]["Origin"].unique()}')
print(f'Unmapped Destinations: {gf_data[gf_data["Dest_Code"].isna()]["Destination"].unique()}')

Sample with codes:
            Origin Origin_Code  Destination Dest_Code    Route  \
0  Fort Lauderdale         FLL     New York       LGA  FLL-LGA   
1  Fort Lauderdale         FLL      Chicago       ORD  FLL-ORD   
2  Fort Lauderdale         FLL      Atlanta       ATL  FLL-ATL   
3  Fort Lauderdale         FLL       Dallas       DFW  FLL-DFW   
4  Fort Lauderdale         FLL  Los Angeles       LAX  FLL-LAX   
5          Orlando         MCO     New York       LGA  MCO-LGA   
6          Orlando         MCO      Chicago       ORD  MCO-ORD   
7          Orlando         MCO    Baltimore       BWI  MCO-BWI   
8        Las Vegas         LAS       Dallas       DFW  LAS-DFW   
9        Las Vegas         LAS  Los Angeles       LAX  LAS-LAX   

   Current_fare_clean   Carrier  
0                99.0     Delta  
1               134.0    United  
2                56.0  Frontier  
3               147.0  Frontier  
4               153.0   JetBlue  
5               108.0  Frontier  
6               

## 7. Calculate Spirit's Average Fare by Route (BTS Q1+Q2 2025)

In [7]:
# Create route identifier for BTS
bts_spirit['Route'] = bts_spirit['ORIGIN'] + '-' + bts_spirit['DEST']

# Calculate average MARKET_FARE per route
# MARKET_FARE appears to be in cents, so divide by 100 to get dollars
bts_spirit['MARKET_FARE_DOLLARS'] = bts_spirit['MARKET_FARE'] / 100

# Group by route and calculate average fare
bts_avg_by_route = bts_spirit.groupby('Route').agg({
    'MARKET_FARE_DOLLARS': 'mean',
    'PASSENGERS': 'sum',
    'ORIGIN': 'first',
    'DEST': 'first',
    'QUARTER': lambda x: ', '.join(map(str, sorted(x.unique())))
}).reset_index()

bts_avg_by_route.columns = ['Route', 'BTS_Avg_Fare', 'Total_Passengers', 'Origin', 'Dest', 'Quarters']
bts_avg_by_route = bts_avg_by_route.sort_values('BTS_Avg_Fare', ascending=False)

print(f'BTS Average Fares by Route (Q1+Q2 2025):')
print(bts_avg_by_route[['Route', 'BTS_Avg_Fare', 'Total_Passengers', 'Quarters']].to_string(index=False))

BTS Average Fares by Route (Q1+Q2 2025):
  Route  BTS_Avg_Fare  Total_Passengers Quarters
ORD-FLL      1.462433            7511.0     1, 2
FLL-ORD      1.393306            7284.0     1, 2
FLL-LGA      1.360496            7928.0     1, 2
DTW-FLL      1.356891            7050.0     1, 2
DTW-MCO      1.302531           10373.0     1, 2
LAS-ATL      1.291366            2820.0     1, 2
MCO-ORD      1.275486            4892.0     1, 2
IAH-FLL      1.196917            5941.0     1, 2
FLL-LAX      1.194517            1114.0     1, 2
LAS-IAH      1.191445            4611.0     1, 2
FLL-DFW      1.170565            4946.0     1, 2
MCO-LGA      1.159653            5716.0     1, 2
IAH-MCO      1.158434            4390.0     1, 2
BWI-FLL      1.113291            6835.0     1, 2
LAS-DFW      1.110524            5692.0     1, 2
MCO-BWI      1.076708            5182.0     1, 2
PHL-MCO      1.066522            5839.0     1, 2
FLL-ATL      0.969489            9247.0     1, 2
ATL-FLL      0.950192       

## 8. Calculate Google Flights Average Fare by Route (May 2026)

In [8]:
# Group by route and calculate average fare
gf_avg_by_route = gf_data.groupby('Route').agg({
    'Current_fare_clean': 'mean',
    'Carrier': lambda x: ', '.join(x.unique()),
    'Origin_Code': 'first',
    'Dest_Code': 'first',
    'Search_Date': 'first',
    'Travel_Date': 'first'
}).reset_index()

gf_avg_by_route.columns = ['Route', 'GF_Avg_Fare', 'Carriers_Available', 'Origin', 'Dest', 'Search_Date', 'Travel_Date']
gf_avg_by_route = gf_avg_by_route.sort_values('GF_Avg_Fare', ascending=False)

print(f'Google Flights Average Fares by Route (May 2026):')
print(gf_avg_by_route[['Route', 'GF_Avg_Fare', 'Carriers_Available']].to_string(index=False))

Google Flights Average Fares by Route (May 2026):
  Route  GF_Avg_Fare Carriers_Available
DTW-MCO        217.0           Frontier
LAS-IAH        179.0           Frontier
LAS-ATL        164.0           Frontier
DTW-FLL        154.0              Delta
PHL-MCO        153.0           Frontier
FLL-LAX        153.0            JetBlue
ORD-FLL        149.0             United
BWI-FLL        148.0           Frontier
FLL-DFW        147.0           Frontier
LAS-DFW        146.0           Frontier
MCO-ORD        142.0           Frontier
FLL-ORD        134.0             United
MCO-BWI        115.0           Frontier
MCO-LGA        108.0           Frontier
FLL-LGA         99.0              Delta
IAH-MCO         99.0           Frontier
LAS-LAX         84.0           Frontier
IAH-FLL         66.0           Frontier
FLL-ATL         56.0           Frontier
ATL-FLL         26.0           Frontier


## 9. Merge BTS & Google Flights Data

In [9]:
# Merge on Route
merged_analysis = bts_avg_by_route[['Route', 'BTS_Avg_Fare', 'Total_Passengers', 'Origin', 'Dest']].merge(
    gf_avg_by_route[['Route', 'GF_Avg_Fare', 'Carriers_Available']],
    on='Route',
    how='inner'
)

print(f'Merged dataset rows: {len(merged_analysis)}')
print(f'\nFirst few rows:')
print(merged_analysis.head())

Merged dataset rows: 20

First few rows:
     Route  BTS_Avg_Fare  Total_Passengers Origin Dest  GF_Avg_Fare  \
0  ORD-FLL      1.462433            7511.0    ORD  FLL        149.0   
1  FLL-ORD      1.393306            7284.0    FLL  ORD        134.0   
2  FLL-LGA      1.360496            7928.0    FLL  LGA         99.0   
3  DTW-FLL      1.356891            7050.0    DTW  FLL        154.0   
4  DTW-MCO      1.302531           10373.0    DTW  MCO        217.0   

  Carriers_Available  
0             United  
1             United  
2              Delta  
3              Delta  
4           Frontier  


## 10. Calculate Fare Impact Metrics

In [10]:
# Calculate fare changes
merged_analysis['Absolute_Change'] = merged_analysis['GF_Avg_Fare'] - merged_analysis['BTS_Avg_Fare']
merged_analysis['Percent_Change'] = ((merged_analysis['GF_Avg_Fare'] - merged_analysis['BTS_Avg_Fare']) / merged_analysis['BTS_Avg_Fare'] * 100).round(2)

# Classify impact level
def classify_impact(pct):
    if pct >= 30:
        return 'High (30%+)'
    elif pct >= 20:
        return 'Medium (20-30%)'
    elif pct >= 10:
        return 'Moderate (10-20%)'
    else:
        return 'Low (<10%)'

merged_analysis['Impact_Level'] = merged_analysis['Percent_Change'].apply(classify_impact)

# Sort by impact
merged_analysis_sorted = merged_analysis.sort_values('Percent_Change', ascending=False)

print('📊 FARE IMPACT ANALYSIS BY ROUTE')
print('=' * 100)
display_cols = ['Route', 'BTS_Avg_Fare', 'GF_Avg_Fare', 'Absolute_Change', 'Percent_Change', 'Impact_Level', 'Carriers_Available']
print(merged_analysis_sorted[display_cols].to_string(index=False))

📊 FARE IMPACT ANALYSIS BY ROUTE
  Route  BTS_Avg_Fare  GF_Avg_Fare  Absolute_Change  Percent_Change Impact_Level Carriers_Available
DTW-MCO      1.302531        217.0       215.697469        16559.87  High (30%+)           Frontier
LAS-IAH      1.191445        179.0       177.808555        14923.78  High (30%+)           Frontier
PHL-MCO      1.066522        153.0       151.933478        14245.70  High (30%+)           Frontier
BWI-FLL      1.113291        148.0       146.886709        13193.91  High (30%+)           Frontier
LAS-DFW      1.110524        146.0       144.889476        13046.95  High (30%+)           Frontier
FLL-LAX      1.194517        153.0       151.805483        12708.53  High (30%+)            JetBlue
LAS-ATL      1.291366        164.0       162.708634        12599.73  High (30%+)           Frontier
FLL-DFW      1.170565        147.0       145.829435        12458.04  High (30%+)           Frontier
LAS-LAX      0.681463         84.0        83.318537        12226.43 

## 11. Calculate KEY PERFORMANCE INDICATORS (KPIs)

In [11]:
# Key metrics for dashboard
avg_percent_change = merged_analysis['Percent_Change'].mean()
median_percent_change = merged_analysis['Percent_Change'].median()
avg_absolute_change = merged_analysis['Absolute_Change'].mean()
total_routes_analyzed = len(merged_analysis)

# Routes by impact tier
high_impact = len(merged_analysis[merged_analysis['Percent_Change'] >= 30])
medium_impact = len(merged_analysis[(merged_analysis['Percent_Change'] >= 20) & (merged_analysis['Percent_Change'] < 30)])
moderate_impact = len(merged_analysis[(merged_analysis['Percent_Change'] >= 10) & (merged_analysis['Percent_Change'] < 20)])
low_impact = len(merged_analysis[merged_analysis['Percent_Change'] < 10])

# Highest and lowest impact routes
highest_impact_route = merged_analysis_sorted.iloc[0]
lowest_impact_route = merged_analysis_sorted.iloc[-1]

# Passenger impact (total passengers affected)
total_passengers_affected = merged_analysis['Total_Passengers'].sum()
avg_passengers_per_route = merged_analysis['Total_Passengers'].mean()

# Revenue impact (hypothetical: if all passengers paid the difference)
merged_analysis['Revenue_Impact'] = merged_analysis['Absolute_Change'] * merged_analysis['Total_Passengers']
total_revenue_impact = merged_analysis['Revenue_Impact'].sum()

print('\n' + '='*80)
print('🎯 HEADLINE METRICS FOR DASHBOARD & PRESENTATION')
print('='*80)
print(f'\n📈 AVERAGE FARE CHANGE:')
print(f'   • Average % increase: {avg_percent_change:.1f}%')
print(f'   • Median % increase: {median_percent_change:.1f}%')
print(f'   • Average $ increase: ${avg_absolute_change:.2f}')

print(f'\n📊 ROUTES BY IMPACT TIER:')
print(f'   • High impact (30%+): {high_impact} routes')
print(f'   • Medium impact (20-30%): {medium_impact} routes')
print(f'   • Moderate impact (10-20%): {moderate_impact} routes')
print(f'   • Low impact (<10%): {low_impact} routes')

print(f'\n🚀 HARDEST HIT ROUTE:')
print(f'   • Route: {highest_impact_route["Route"]}')
print(f'   • BTS Avg Fare (Q1-Q2 2025): ${highest_impact_route["BTS_Avg_Fare"]:.2f}')
print(f'   • Google Flights (May 2026): ${highest_impact_route["GF_Avg_Fare"]:.2f}')
print(f'   • % Increase: {highest_impact_route["Percent_Change"]:.1f}%')
print(f'   • $ Increase: ${highest_impact_route["Absolute_Change"]:.2f}')

print(f'\n✅ LOWEST IMPACT ROUTE:')
print(f'   • Route: {lowest_impact_route["Route"]}')
print(f'   • % Change: {lowest_impact_route["Percent_Change"]:.1f}%')
print(f'   • $ Change: ${lowest_impact_route["Absolute_Change"]:.2f}')

print(f'\n👥 PASSENGER IMPACT:')
print(f'   • Total passengers affected (Q1-Q2 2025): {total_passengers_affected:,.0f}')
print(f'   • Average per route: {avg_passengers_per_route:,.0f}')

print(f'\n💰 ESTIMATED REVENUE IMPACT:')
print(f'   • Total (if all passengers paid difference): ${total_revenue_impact:,.0f}')
print(f'   • Per route average: ${merged_analysis["Revenue_Impact"].mean():,.0f}')
print(f'   • Annualized estimate (4 quarters): ${total_revenue_impact * 4:,.0f}')

print('\n' + '='*80)


🎯 HEADLINE METRICS FOR DASHBOARD & PRESENTATION

📈 AVERAGE FARE CHANGE:
   • Average % increase: 10649.7%
   • Median % increase: 11141.2%
   • Average $ increase: $125.78

📊 ROUTES BY IMPACT TIER:
   • High impact (30%+): 20 routes
   • Medium impact (20-30%): 0 routes
   • Moderate impact (10-20%): 0 routes
   • Low impact (<10%): 0 routes

🚀 HARDEST HIT ROUTE:
   • Route: DTW-MCO
   • BTS Avg Fare (Q1-Q2 2025): $1.30
   • Google Flights (May 2026): $217.00
   • % Increase: 16559.9%
   • $ Increase: $215.70

✅ LOWEST IMPACT ROUTE:
   • Route: ATL-FLL
   • % Change: 2636.3%
   • $ Change: $25.05

👥 PASSENGER IMPACT:
   • Total passengers affected (Q1-Q2 2025): 121,806
   • Average per route: 6,090

💰 ESTIMATED REVENUE IMPACT:
   • Total (if all passengers paid difference): $14,914,060
   • Per route average: $745,703
   • Annualized estimate (4 quarters): $59,656,240



## 12. Carrier Analysis - Who Filled the Gap?

In [12]:
# Extract individual carriers from the comma-separated list
from collections import Counter

# Count carriers serving Spirit's routes
all_carriers = []
for carrier_list in gf_avg_by_route['Carriers_Available']:
    carriers = [c.strip() for c in carrier_list.split(',')]
    all_carriers.extend(carriers)

carrier_counts = Counter(all_carriers)
carrier_df = pd.DataFrame.from_dict(carrier_counts, orient='index', columns=['Route_Count']).reset_index()
carrier_df.columns = ['Carrier', 'Routes_Served']
carrier_df = carrier_df.sort_values('Routes_Served', ascending=False)

print('\n📍 CARRIERS SERVING SPIRIT\'S 20 ROUTES (May 2026):')
print('='*50)
print(carrier_df.to_string(index=False))

print(f'\n🎯 KEY INSIGHT: Frontier serves {carrier_df[carrier_df["Carrier"] == "Frontier"]["Routes_Served"].values[0]} of {total_routes_analyzed} routes!')


📍 CARRIERS SERVING SPIRIT'S 20 ROUTES (May 2026):
 Carrier  Routes_Served
Frontier             15
   Delta              2
  United              2
 JetBlue              1

🎯 KEY INSIGHT: Frontier serves 15 of 20 routes!


## 13. Export Analysis for Dashboard/Visualization

In [13]:
# Export full analysis
merged_analysis_sorted.to_csv(data_path / 'fare_impact_analysis.csv', index=False)
carrier_df.to_csv(data_path / 'carrier_analysis.csv', index=False)

print('✅ Exported:')
print('   • fare_impact_analysis.csv - Full route-level analysis')
print('   • carrier_analysis.csv - Carrier market share')

# Create summary stats for easy reference
summary_kpis = {
    'metric': [
        'Average Fare Increase (%)',
        'Median Fare Increase (%)',
        'Average Fare Increase ($)',
        'Routes Analyzed',
        'Routes with 30%+ Increase',
        'Routes with 20-30% Increase',
        'Routes with 10-20% Increase',
        'Routes with <10% Increase',
        'Highest Impact Route',
        'Highest Impact %',
        'Total Passengers Affected (Q1-Q2)',
        'Total Revenue Impact ($)',
        'Top Carrier Filling Gap',
        'Top Carrier Route Count'
    ],
    'value': [
        f'{avg_percent_change:.1f}%',
        f'{median_percent_change:.1f}%',
        f'${avg_absolute_change:.2f}',
        total_routes_analyzed,
        high_impact,
        medium_impact,
        moderate_impact,
        low_impact,
        highest_impact_route['Route'],
        f'{highest_impact_route["Percent_Change"]:.1f}%',
        f'{total_passengers_affected:,.0f}',
        f'${total_revenue_impact:,.0f}',
        'Frontier',  # from your observation
        carrier_df.iloc[0]['Routes_Served']
    ]
}

summary_df = pd.DataFrame(summary_kpis)
summary_df.to_csv(data_path / 'kpi_summary.csv', index=False)

print('   • kpi_summary.csv - All KPIs for reference')
print('\n✨ Analysis complete! Ready for dashboard & deck.')

✅ Exported:
   • fare_impact_analysis.csv - Full route-level analysis
   • carrier_analysis.csv - Carrier market share
   • kpi_summary.csv - All KPIs for reference

✨ Analysis complete! Ready for dashboard & deck.
